# MAINTAIN AI V1.1 — Leakage-Safe Temporal Bootstrap

This run fixes the two V1 experimental issues: (1) the 70/15/15 asset split is actually used for training/validation/internal test, and (2) normalization statistics are fitted only on the training partition and then applied unchanged to validation/internal test/official C-MAPSS test. The model remains a C-MAPSS RUL bootstrap model; 24 steps are C-MAPSS cycles, not hours, and the 24h/48h/7d risk heads remain untrained.

In [ ]:
!git clone -b Lab https://github.com/jadhavdurvesh/Maintain.ai.3.git /content/Maintain.ai.3
%cd /content/Maintain.ai.3
!pip install -q -r training/requirements.txt

In [ ]:
import os, random, numpy as np, torch
os.chdir('/content')
random.seed(42); np.random.seed(42); torch.manual_seed(42)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## 1. C-MAPSS input

Upload `CMAPSSData.zip` from the official NASA C-MAPSS dataset if it is not already present. This notebook intentionally does not manufacture telemetry.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
for name in uploaded: print('Uploaded:', name)

In [ ]:
import zipfile, re, pandas as pd
from pathlib import Path
ROOT=Path('/content'); RAW=ROOT/'cmapss_raw'; PREP=ROOT/'prepared/cmapss'; PREP.mkdir(parents=True,exist_ok=True)
zips=list(ROOT.glob('*.zip'))
if not zips: raise FileNotFoundError('Upload CMAPSSData.zip first.')
with zipfile.ZipFile(zips[0]) as z: z.extractall(RAW)
files_all=list(RAW.rglob('*'))
def find(name):
    hits=[p for p in files_all if p.name==name]
    if not hits: raise FileNotFoundError(name)
    return hits[0]
def read_train(path, subset):
    cols=['unit','cycle','op_1','op_2','op_3']+[f'sensor_{i}' for i in range(1,22)]
    d=pd.read_csv(path,sep=r'\\s+',header=None,names=cols,engine='python')
    d['rul_steps']=d.groupby('unit')['cycle'].transform('max')-d['cycle']
    d['asset_id']=[f'{subset}_train_{u:04d}' for u in d.unit]
    d['timestamp']=d['cycle'].astype(float); d['category']='other'; return d
def read_test(path,rul_path,subset):
    cols=['unit','cycle','op_1','op_2','op_3']+[f'sensor_{i}' for i in range(1,22)]
    d=pd.read_csv(path,sep=r'\\s+',header=None,names=cols,engine='python')
    rul=pd.read_csv(rul_path,sep=r'\\s+',header=None)[0].astype(float).to_numpy()
    ends=d.groupby('unit')['cycle'].max().sort_index(); targets=dict(zip(ends.index,rul))
    d['rul_steps']=[targets[u]+ends[u]-c for u,c in zip(d.unit,d.cycle)]
    d['asset_id']=[f'{subset}_test_{u:04d}' for u in d.unit]
    d['timestamp']=d['cycle'].astype(float); d['category']='other'; return d
train=pd.concat([read_train(find(f'train_{s}.txt'),s) for s in ['FD001','FD002','FD003','FD004']],ignore_index=True)
test=pd.concat([read_test(find(f'test_{s}.txt'),find(f'RUL_{s}.txt'),s) for s in ['FD001','FD002','FD003','FD004']],ignore_index=True)
train.to_parquet(PREP/'train_all.parquet',index=False); test.to_parquet(PREP/'test_all.parquet',index=False)
print('Train:',len(train),'rows /',train.asset_id.nunique(),'engines'); print('Test:',len(test),'rows /',test.asset_id.nunique(),'engines')

In [ ]:
# 2. Deterministic asset split + training-only normalization
import hashlib, json, numpy as np, pandas as pd
from pathlib import Path
TRAIN= pd.read_parquet('/content/prepared/cmapss/train_all.parquet')
def split_of(asset_id):
    h=int(hashlib.sha256(f'42:{asset_id}'.encode()).hexdigest()[:8],16)%100
    return 'test' if h<15 else ('val' if h<30 else 'train')
TRAIN['split']=TRAIN.asset_id.map(split_of)
SPLIT_DIR=Path('/content/splits/cmapss'); SPLIT_DIR.mkdir(parents=True,exist_ok=True)
for s in ['train','val','test']:
    TRAIN[TRAIN.split==s].drop(columns='split').to_parquet(SPLIT_DIR/f'{s}.parquet',index=False)
sets={s:set(TRAIN.loc[TRAIN.split==s,'asset_id']) for s in ['train','val','test']}
assert not (sets['train']&sets['val'] or sets['train']&sets['test'] or sets['val']&sets['test'])
print({s:len(v) for s,v in sets.items()})
CHANNELS=['op_1','op_2','op_3']+[f'sensor_{i}' for i in range(1,22)]
fit=TRAIN[TRAIN.split=='train'][CHANNELS].to_numpy(dtype=np.float64)
mean=np.nanmean(fit,axis=0); std=np.nanstd(fit,axis=0); std=np.where(std<1e-8,1.0,std)
NORM={'channels':CHANNELS,'mean':mean.tolist(),'std':std.tolist(),'fit_scope':'training_assets_only','seed':42}
Path('/content/splits/cmapss/normalization.json').write_text(json.dumps(NORM,indent=2))
print('Normalization fitted on',len(sets['train']),'training engines only.')

In [ ]:
# 3. Build leakage-safe sequences for train/val/internal-test and official NASA test
import numpy as np, pandas as pd, torch
from pathlib import Path
SEQ=24; out=Path('/content/sequences_v1_1'); out.mkdir(parents=True,exist_ok=True)
mean=np.array(NORM['mean'],dtype=np.float32); std=np.array(NORM['std'],dtype=np.float32)
def vec(row): return row[CHANNELS].to_numpy(dtype=np.float32)
def build(df, endpoint_only=False):
    xs=[]; cats=[]; ruls=[]; ids=[]
    for aid,g in df.groupby('asset_id',sort=False):
        g=g.sort_values('timestamp').reset_index(drop=True)
        if len(g)<SEQ: continue
        values=(np.stack([vec(g.iloc[i]) for i in range(len(g))])-mean)/std
        ends=[len(g)-1] if endpoint_only else range(SEQ-1,len(g))
        for e in ends:
            xs.append(values[e-SEQ+1:e+1]); cats.append(4); ruls.append(float(g.iloc[e].rul_steps)); ids.append(aid)
    return {'x':torch.tensor(np.stack(xs),dtype=torch.float32),'category_id':torch.tensor(cats), 'rul_target':torch.tensor(ruls,dtype=torch.float32),'asset_id':ids}
for s in ['train','val','test']:
    d=pd.read_parquet(f'/content/splits/cmapss/{s}.parquet'); torch.save(build(d),out/f'cmapss_{s}.pt'); print(s,len(d.asset_id.unique()),'assets ->',len(torch.load(out/f'cmapss_{s}.pt',weights_only=False)['x']))
official=pd.read_parquet('/content/prepared/cmapss/test_all.parquet'); torch.save(build(official,endpoint_only=True),out/'cmapss_official_test.pt')
print('Official test sequences:',len(torch.load(out/'cmapss_official_test.pt',weights_only=False)['x']))

In [ ]:
# 4. Train with validation + early stopping
import os, torch, torch.nn as nn, torch.nn.functional as F
os.chdir('/content')
from torch.utils.data import TensorDataset,DataLoader
class SharedTemporalModel(nn.Module):
    def __init__(self):
        super().__init__(); self.conv=nn.Sequential(nn.Conv1d(28,64,5,padding=2),nn.BatchNorm1d(64),nn.GELU(),nn.Conv1d(64,96,5,padding=2),nn.BatchNorm1d(96),nn.GELU()); self.gru=nn.GRU(96,128,2,batch_first=True,dropout=.1); self.emb=nn.Embedding(5,16); self.fuse=nn.Sequential(nn.Linear(144,128),nn.GELU(),nn.Dropout(.1)); self.risk=nn.Linear(128,3); self.rul=nn.Sequential(nn.Linear(128,64),nn.GELU(),nn.Linear(64,1))
    def forward(self,x,c):
        x=self.conv(x.transpose(1,2)).transpose(1,2); h,_=self.gru(x); z=self.fuse(torch.cat([h[:,-1],self.emb(c)],1)); r=F.softplus(self.rul(z)); return self.risk(z),r.squeeze(1)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(device)
def loadpt(n): return torch.load(f'/content/sequences_v1_1/cmapss_{n}.pt',weights_only=False)
tr,va=loadpt('train'),loadpt('val'); train_ds=TensorDataset(tr['x'],tr['category_id'],tr['rul_target']); val_ds=TensorDataset(va['x'],va['category_id'],va['rul_target'])
tl=DataLoader(train_ds,batch_size=128,shuffle=True,num_workers=0,pin_memory=device.type=='cuda'); vl=DataLoader(val_ds,batch_size=256,shuffle=False,num_workers=0)
model=SharedTemporalModel().to(device); opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4); best=float('inf'); best_state=None; patience=7; stale=0; history=[]
for epoch in range(1,51):
    model.train(); total=n=0
    for bx,bc,by in tl:
        bx,bc,by=bx.to(device),bc.to(device),by.to(device); opt.zero_grad(set_to_none=True); _,pred=model(bx,bc); loss=F.smooth_l1_loss(pred,by); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); total+=loss.item()*len(bx); n+=len(bx)
    model.eval(); vt=vn=0
    with torch.no_grad():
        for bx,bc,by in vl:
            _,pred=model(bx.to(device),bc.to(device)); loss=F.smooth_l1_loss(pred,by.to(device)); vt+=loss.item()*len(bx); vn+=len(bx)
    tr_loss=total/n; va_loss=vt/vn; history.append([epoch,tr_loss,va_loss]); print(f'Epoch {epoch:02d} | train {tr_loss:.4f} | val {va_loss:.4f}')
    if va_loss<best-1e-4: best=va_loss; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
    else: stale+=1
    if stale>=patience: print('Early stopping'); break
model.load_state_dict(best_state); ART=Path('/content/artifacts_v1_1'); ART.mkdir(exist_ok=True)
torch.save({'model_state_dict':model.state_dict(),'model_version':'shared-temporal-v1.1','input_channels':28,'category_count':5,'sequence_length':24,'hidden_size':128,'category_embedding_size':16,'bootstrap_dataset':'NASA C-MAPSS FD001-FD004','training_objective':'RUL','normalization':NORM,'risk_status':'not_trained'},ART/'shared_temporal_v1_1.pt')
json.dump({'history':history,'best_val_loss':best},open(ART/'training_history.json','w'),indent=2)

In [ ]:
# 5. Evaluate internal test and official NASA test
import numpy as np, torch, json
def eval_file(path):
    d=torch.load(path,weights_only=False); ds=TensorDataset(d['x'],d['category_id'],d['rul_target']); dl=DataLoader(ds,batch_size=256); ps=[]; ys=[]
    model.eval()
    with torch.no_grad():
        for x,c,y in dl: ps.append(model(x.to(device),c.to(device))[1].cpu().numpy()); ys.append(y.numpy())
    p=np.concatenate(ps); y=np.concatenate(ys); e=p-y
    return {'samples':int(len(y)),'assets':int(len(set(d['asset_id']))),'mae':float(np.mean(np.abs(e))),'rmse':float(np.sqrt(np.mean(e**2))),'actual_min':float(y.min()),'actual_max':float(y.max())}
internal=eval_file('/content/sequences_v1_1/cmapss_test.pt'); official=eval_file('/content/sequences_v1_1/cmapss_official_test.pt'); results={'internal_test':internal,'official_nasa_test':official,'model_version':'shared-temporal-v1.1'}; print(json.dumps(results,indent=2)); json.dump(results,open('/content/artifacts_v1_1/evaluation.json','w'),indent=2)

In [ ]:
# 6. Export reproducibility bundle
import zipfile, shutil
B=Path('/content/maintain_ai_shared_temporal_v1_1'); shutil.rmtree(B,ignore_errors=True); B.mkdir()
for p in Path('/content/artifacts_v1_1').glob('*'): shutil.copy2(p,B/p.name)
with zipfile.ZipFile('/content/maintain_ai_shared_temporal_v1_1.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in B.rglob('*'): z.write(p,p.relative_to(B.parent))
print('/content/maintain_ai_shared_temporal_v1_1.zip')

In [ ]:
from google.colab import files
files.download('/content/maintain_ai_shared_temporal_v1_1.zip')

## V1.1 interpretation

Report internal-test metrics separately from the official NASA C-MAPSS test. Do not compare V1 and V1.1 as a quality claim unless the preprocessing, split, and evaluation protocol are identical. The production 24h/48h/7d risk task is still untrained and requires timestamp-aware MAINTAIN AI telemetry plus technician-confirmed outcomes.